# Phase 1 Results Review

Use this notebook to compare prompt-only, repair, full-schema QLoRA, and reduced-schema QLoRA results in one place.

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
METRICS_DIR = PROJECT_ROOT / 'results' / 'metrics'
METRICS_DIR


In [ ]:
REPORTS = {
    'prompt_only': METRICS_DIR / 'qwen25_3b_prompt_only_test_report.json',
    'prompt_only_repair': METRICS_DIR / 'qwen25_3b_prompt_only_test_repaired_report.json',
    'qlora_full': METRICS_DIR / 'qwen25_3b_phase1_qlora_v1_test_report.json',
    'qlora_full_repair': METRICS_DIR / 'qwen25_3b_phase1_qlora_v1_test_repaired_report.json',
    'qlora_reduced': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_v1_test_report.json',
    'qlora_reduced_repair': METRICS_DIR / 'qwen25_3b_phase1_qlora_reduced_v1_test_repaired_report.json',
}

def load_report(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

loaded = {name: load_report(path) for name, path in REPORTS.items()}
missing = [name for name, report in loaded.items() if report is None]
print('missing_reports =', missing)


In [ ]:
summary_rows = []
for name, report in loaded.items():
    if report is None:
        continue
    summary = report['summary']
    summary_rows.append({
        'experiment': name,
        'valid_json_rate': round(summary['valid_json_rate'], 4),
        'schema_compliance_rate': round(summary['schema_compliance_rate'], 4),
        'field_exact_match': round(summary['field_exact_match'], 4),
        'end_to_end_exact_match': round(summary['end_to_end_exact_match'], 4),
        'primary_error_counts': summary['error_counts'],
    })

summary_rows


In [ ]:
for key in ['qlora_reduced', 'qlora_reduced_repair']:
    report = loaded.get(key)
    if report is None:
        continue
    print(f'=== {key} by complexity ===')
    print(json.dumps(report.get('grouped_summary', {}).get('by_complexity_bucket', {}), indent=2, ensure_ascii=False))
    print()


In [ ]:
FINDINGS_PATH = PROJECT_ROOT / 'docs' / 'results' / 'phase1_baseline_findings.md'
print(FINDINGS_PATH.read_text(encoding='utf-8'))
